# Mining a Causal Graph from a Corpus with `causalatee.mining`

`causalatee.mining` turns a corpus of raw documents into an aggregated causal graph: each document
flows through detection/candidate-extraction/identification stages, and the resulting relations are
reduced into a `causalatee.graph.Graph` by `(cause, effect)` pair. This is the same *kind* of
resource CauseNet or CausalBank represent, built from your own corpus instead of an existing one.

This notebook demonstrates the pipeline mechanics with a small, deterministic stand-in extraction
function instead of a real model, so it runs without downloading any weights or needing a GPU. A
real pipeline plugs in `causalatee.models` Protocol implementations (`Detection`,
`CandidateExtraction`, `PairwiseIdentification`) at the `.filter`/`.map` stages instead -- see the
[API reference](../reference/mining.md) for the full stage-by-stage design.

## Setup

`causalatee.mining` needs the `mining` extra (`aiostream`).

In [ ]:
%pip install -q "causalatee[mining]"

## Define a document source and a toy extraction function

`Pipeline` consumes any `AsyncIterable[Document]`. Real corpora would stream from a database, an
API, or a file on disk; here, four documents are yielded directly. The stand-in extraction function
returns causalatee's `ExtractedRelation` shape (`{"e1", "e2", "relation", "score"}`) -- the same
shape `causalatee.models.compose_extraction` produces from real Detection/CandidateExtraction/
PairwiseIdentification models.

In [ ]:
from causalatee.mining import Document, Pipeline, graph_sink


async def source():
    yield Document(id="1", text="The storm caused severe flooding across the region.")
    yield Document(id="2", text="Analysts say the storm caused flooding downtown.")
    yield Document(id="3", text="Sugar does not cause hyperactivity in children.")
    yield Document(id="4", text="Rising sea levels are driven by climate change.")


def toy_extraction(doc):
    text = doc.text.lower()
    if "storm" in text and "flooding" in text:
        return [{"e1": "storm", "e2": "flooding", "relation": "Causal", "score": 0.92}]
    if "sugar" in text and "hyperactivity" in text:
        return [{"e1": "sugar", "e2": "hyperactivity", "relation": "Countercausal", "score": 0.81}]
    if "sea levels" in text and "climate change" in text:
        return [{"e1": "climate change", "e2": "sea levels", "relation": "Causal", "score": 0.88}]
    return []

## Run the pipeline

`.map(toy_extraction, concurrency=1)` runs the extraction stage; `.reduce(graph_sink())` drains the
pipeline into a `GraphSink`, which spools every relation unaggregated and only aggregates lazily on
first access (see the [API reference](../reference/mining.md) for why). Document 1 and 2 both
mention `storm -> flooding`, so that pair's `support` should end up as 2, not two separate edges.

In [ ]:
graph = await Pipeline(source()).map(toy_extraction, concurrency=1).reduce(graph_sink())

# Not closed yet -- inspect first, close only once fully done with it (below).
print(f"{len(graph.nodes)} nodes, {len(graph.edges)} edges")
for edge in sorted(graph.edges, key=lambda e: e.source.id):
    print(f"{edge.source.id!r} -> {edge.target.id!r}: {dict(edge.metadata)}")

Expected output:
```
6 nodes, 3 edges
'climate change' -> 'sea levels': {'support': 1, 'causal_count': 1, 'countercausal_count': 0, 'avg_score': 0.88}
'storm' -> 'flooding': {'support': 2, 'causal_count': 2, 'countercausal_count': 0, 'avg_score': 0.92}
'sugar' -> 'hyperactivity': {'support': 1, 'causal_count': 0, 'countercausal_count': 1, 'avg_score': 0.81}
```

`storm -> flooding` aggregated the two mentions into one edge (`support=2`), and the countercausal
mention of `sugar -> hyperactivity` is a real `Relation.Countercausal`, not folded into the causal
count -- `causal_count`/`countercausal_count` are tracked separately, matching how
`causalatee.graph.CauseEffectGraph` and CCNC both distinguish the two.

## Persist the mined graph as CGF

`GraphSink` *is* a `causalatee.graph.Graph`, so it can be passed to `save_cgf` directly, no
conversion step needed -- the same as any other `Graph` implementation in this package.

In [ ]:
from causalatee.graph import load_cgf, save_cgf

with graph:
    save_cgf(graph, "mined.cgf")

with load_cgf("mined.cgf", validate=True) as mapped:
    storm = mapped.get_node("storm")
    (out_edge,) = list(storm.outgoing_edges())
    print(f"{out_edge.source.id!r} -> {out_edge.target.id!r}: {dict(out_edge.metadata)}")

Expected output:
```
'storm' -> 'flooding': {'support': 2, 'causal_count': 2, 'countercausal_count': 0, 'avg_score': 0.92}
```

`with graph:` closes `GraphSink`'s temporary spool database once the block exits -- always close a
`GraphSink` (or use it as a context manager, as above) once you're fully done reading from it, not
before -- closing ends its usable lifetime, including for `save_cgf`.